# ارزیابی شدت v2 روی Colab — ۶ run کامل (پس از اصلاح‌های audit)

همان پروتکل finding 24 (`severity_sweep_patch_v2`, S1–S6 × {acute, persistent} × {ndvi, savi},
rounds=2) برای هر ۶ آموزش v2: `patch_pixelwise` و `patch_cnn_gru` × seeds `13, 42, 2026`.

**چه چیزی لازم است:**
1. `thesis_data_full_v2.zip` (~۲۹۴ MB) — از قبل در Drive هست (نوتبوک آموزش)
2. `patch_full_v2_eval_bundle.zip` (**۴٫۵ MB** — همین حالا آپلود کنید → `MyDrive/thesis_colab/`)
   شامل cycles parquet + ۶ پوشهٔ checkpoint (last.pt + norm + sigma)
3. Runtime → **T4 GPU** (اگر T4 نبود، CPU هم کار می‌کند ولی کندتر است)

هر run مستقل است؛ اگر session قطع شد، سلول ارزیابی را دوباره اجرا کنید —
runهای تمام‌شده با `metrics_*.json` در `reports/dev/patch_full_v2_ep10/` هستند و
سلول ارزیابی آن‌ها را skip می‌کند.

In [ ]:
import subprocess, sys, torch

print("python:", sys.version.split()[0], "| torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "zarr==3.3.0", "lightning", "click"], check=True)
import zarr
print("zarr:", zarr.__version__)

In [ ]:
import subprocess

REPO_URL = "https://github.com/Mahdikalantari555/thesis_implementation.git"
r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL,
                    "/content/thesis_implementation"])
if r.returncode != 0:
    raise SystemExit("clone ناموفق — ریپو باید public و push شده باشد")
head = subprocess.run(["git", "-C", "/content/thesis_implementation", "rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("HEAD:", head)
print("*(این handover روی commit c5fd082 ثبت شده — فلگ --splits و پنل v2 باید داخل HEAD باشد)*")

In [ ]:
from google.colab import drive
import pathlib, shutil, zipfile

drive.mount("/content/drive")
REPO = "/content/thesis_implementation"

for name in ("thesis_data_full_v2.zip", "patch_full_v2_eval_bundle.zip"):
    src = pathlib.Path(f"/content/drive/MyDrive/thesis_colab/{name}")
    assert src.is_file(), f"{name} در Drive پیدا نشد — آن را در MyDrive/thesis_colab/ بگذارید"
    shutil.copy(src, f"/content/{name}")

with zipfile.ZipFile("/content/thesis_data_full_v2.zip") as z:
    z.extractall(REPO)
    print("data zip:", len(z.namelist()), "entries extracted")
with zipfile.ZipFile("/content/eval_bundle.zip") as z:
    z.extractall(REPO)
    print("eval bundle:", len(z.namelist()), "entries extracted")

In [ ]:
import os, sys, json, hashlib, pathlib

os.chdir("/content/thesis_implementation")
sys.path.insert(0, "/content/thesis_implementation/src")

import zarr
from ssl_anomaly.io import patches as P
from ssl_anomaly.monitor import splits as S

man = S.load_splits_locked(path="data/processed/splits_locked_v2.json", verify_cycle_files=False)
root = zarr.open_group("data/patch_runs/v3/patches.zarr", mode="r")
P.validate_store(root, expected_fields=490)
prov = json.loads(open("data/patch_runs/v3/patches.zarr/provenance.json").read())
assert prov["splits_manifest_sha256"] == man["manifest_sha256"]
pq_hash = hashlib.sha256(open("data/processed/cycles_490_v2.parquet", "rb").read()).hexdigest()
assert pq_hash == man["cycle_file_hashes"]["cycles_parquet_sha256"], "parquet با manifest یکی نیست"
assert int(root["cycle_valid"][:].sum()) == 3478
print("gates OK — store", prov["git_sha"][:7], "| splits", man["manifest_sha256"][:8],
      "| parquet", pq_hash[:8])

In [ ]:
import glob, os, subprocess, sys, time

os.chdir("/content/thesis_implementation")
env = dict(os.environ, PYTHONPATH="/content/thesis_implementation/src")
runs = [f"{a}_seed{s}" for a in ("patch_pixelwise", "patch_cnn_gru") for s in (13, 42, 2026)]
for run in runs:
    metric = f"reports/dev/patch_full_v2_ep10/metrics_{run.split('_seed')[0]}_sev_{run.split('_seed')[0]}_s{run.split('_seed')[1]}.json"
    if os.path.exists(metric):
        print(f"=== {run}: metrics موجود است، skip ===")
        continue
    d = f"checkpoints/patch/patch_full_v2_ep10/{run}"
    base = ["-X", "utf8", "-u", "scripts/eval_dev.py",
            "--run-id", "patch_full_v2_ep10",
            "--model", run.split("_seed")[0], "--scorer", run.split("_seed")[0],
            "--protocol", "severity",
            "--patch-store", "data/patch_runs/v3/patches.zarr",
            "--patch-checkpoint", f"{d}/last.pt",
            "--patch-norm", f"{d}/norm_stats.json",
            "--patch-sigma", f"{d}/patch_sigma.npz",
            "--splits", "data/processed/splits_locked_v2.json",
            "--holdout-manifest", "data/processed/holdout_manifest_v2.json",
            "--seeds", run.split("_seed")[1], "--rounds", "2"]
    print(f"=== {run} start {time.strftime('%H:%M:%S')} ===", flush=True)
    t0 = time.time()
    # T4 → cuda؛ اگر خطای cuda داد، همان run روی cpu تکرار می‌شود
    r = subprocess.run([sys.executable, *base, "--device", "cuda"], env=env)
    if r.returncode != 0:
        print(f"cuda ناموفق (exit {r.returncode}) — تلاش با cpu", flush=True)
        r = subprocess.run([sys.executable, *base, "--device", "cpu"], env=env)
    print(f"=== {run} exit={r.returncode} wall={time.time()-t0:.0f}s ===", flush=True)
    assert os.path.exists(metric), f"metrics نوشته نشد: {metric}"
print(f"=== ALL 6 EVALS DONE {time.strftime('%H:%M:%S')} ===")

In [ ]:
import json, os, shutil, zipfile, glob

os.chdir("/content/thesis_implementation")
metrics = sorted(glob.glob("reports/dev/patch_full_v2_ep10/metrics_*.json"))
assert len(metrics) == 6, f"توضعات ۶ metrics بود، پیدا شد: {len(metrics)}"
dest = "/content/drive/MyDrive/thesis_colab/patch_full_v2_sigma_eval_colab.zip"
with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as z:
    for f in metrics + sorted(glob.glob("reports/dev/patch_full_v2_ep10/progress_eval_*.jsonl")):
        z.write(f, f)
        print("bundled:", f)
print("به Drive برگشت:", dest, f"({os.path.getsize(dest)/1e3:.0f} KB)")

## بعد از اتمام

* zip نتایج را به ایجنت محلی بدهید — راستی‌آزمایی ۶ metrics → جدول MDM80 → finding 25 (v2)
  → مقایسهٔ v1↔v2 (task 5.3).
* یادآوری: اعداد v2 با injection باند-سازگار + scoring one-step-ahead (اصلاح‌های audit) و دادهٔ v2
  تولید می‌شوند؛ مقایسه با v1 باید این تغییر پروتکل را صادقانه ذکر کند.
* device هر run داخل metrics ثبت می‌شود (cuda/cpu)؛ float جزئی بین دو device فرق می‌کند.